## homework 4

In [2]:
import torch
import torch.nn as nn

from torch.nn import Sequential
from torch.nn import GRU, LSTM
from torch.utils.data import DataLoader, TensorDataset
from torch import tensor

import numpy as np
import pandas as pd
import yfinance

import time

from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import shuffle

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


1. download and preprocess the data
- sliding window to collect sequences


TODO may need to go back and check for bad values that are not null. 

In [3]:
# do this here if needed... download for 2021
# 80/20 split for data

# TODO 
# ensure the preprocessing is correct
# ask about the shuffling 
# see if the modeling is correct 

M = 60
N = 1


# S&P, KO, WALMART, Edison Utility Co. 

rstate = np.random.RandomState(1)


def preprocess(data : pd.DataFrame, batch_size_ = 10):
    data_close = data["Close"].to_numpy()
    X_seq = []
    y_seq = []
    seqs = []

    # use sliding window to create sequences
    # TODO check the indices!
    for i in range(len(data_close) - M - N):
        X_seq.append(data_close[i : i + M])
        y_seq.append(data_close[i + M])

    print("X_seq length:", len(X_seq))
    print("y_seq length:", len(y_seq))

    # remove extra dimension placed in. 
    X_seq_np = np.array(X_seq).squeeze()
    y_seq_np = np.array(y_seq).squeeze()

    print("X_seq_np shape: ", X_seq_np.shape)

    # use sklearn to shuffle both and keep indices the same
    # this apparently does not pattern match. 
    shuffled = shuffle(X_seq_np, y_seq_np, random_state=rstate)
    print(len(shuffled))
    X_shuffle = shuffled[0]
    y_shuffle = shuffled[1]

    print("X_shuffle shape: ", X_shuffle.shape)


    # split 80/20
    split_index = int(np.size(X_shuffle, axis = 0) * 0.8)
    X_train = X_shuffle[:split_index, :]
    X_test = X_shuffle[split_index:, :]
    y_train = y_shuffle[:split_index]
    y_test = y_shuffle[split_index:]

    print("X_train shape: ", np.shape(X_train))
    print("y_train shape: ", np.shape(y_train))
    print("X_test shape: ", np.shape(X_test))
    print("y_test shape: ", np.shape(y_test))

    # Scale the data
    # need to scale the labels too since we are using an activation function
    # need to add an extra dimension because sklearn is expecting a 2D matrix. 
    mms_X = MinMaxScaler()
    mms_y = MinMaxScaler()
    X_train = mms_X.fit_transform(X_train)
    y_train = mms_y.fit_transform(y_train.reshape(-1, 1)).squeeze()
    X_test = mms_X.transform(X_test)
    y_test  = mms_y.transform(y_test.reshape(-1, 1)).squeeze()

    # # convert to tensors 
    # X_train_tensor = torch.tensor(X_train, dtype = torch.float64)
    # y_train_tensor = torch.tensor(y_train, dtype = torch.float64)
    # X_test_tensor = torch.tensor(X_test, dtype = torch.float64)
    # y_test_tensor = torch.tensor(y_test, dtype = torch.float64)

    # # put inside tensor datasets
    # train_set = TensorDataset(X_train_tensor, y_train_tensor)
    # test_set = TensorDataset(X_test_tensor, y_test_tensor)

    # return [train_set, test_set]

    # a dict is much smarter here than a list I should have done this sooner
    return {
        "X_train" : X_train, 
        "y_train" : y_train, 
        "X_test" : X_test, 
        "y_test" :  y_test
    }

data_spy = yfinance.download('SPY', start = '2020-01-01', end = '2022-01-01')
data_ko = yfinance.download('KO', start = '2020-01-01', end = '2022-01-01')
data_wmt = yfinance.download('WMT', start = '2020-01-01', end = '2022-01-01')
data_ed = yfinance.download('ED', start = '2020-01-01', end = '2022-01-01')

stocks = [data_spy, data_ko, data_wmt, data_ed]
prepped_stocks = []

for stock in stocks:
    prepped_stocks.append(preprocess(stock))

# nvda_ten = torch.tensor(data_nvda, dtype=np.float64);
# gme_ten = torch.tensor(data_gme, dtype=np.float64);
# dji_ten = torch.tensor(data_dji, dtype=np.float64);
# ma_ten = torch.tensor(data_ma, dtype=np.float64);


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

X_seq length: 444
y_seq length: 444
X_seq_np shape:  (444, 60)
2
X_shuffle shape:  (444, 60)
X_train shape:  (355, 60)
y_train shape:  (355,)
X_test shape:  (89, 60)
y_test shape:  (89,)
X_seq length: 444
y_seq length: 444
X_seq_np shape:  (444, 60)
2
X_shuffle shape:  (444, 60)
X_train shape:  (355, 60)
y_train shape:  (355,)
X_test shape:  (89, 60)
y_test shape:  (89,)
X_seq length: 444
y_seq length: 444
X_seq_np shape:  (444, 60)
2
X_shuffle shape:  (444, 60)
X_train shape:  (355, 60)
y_train shape:  (355,)
X_test shape:  (89, 60)
y_test shape:  (89,)
X_seq length: 444
y_seq length: 444
X_seq_np shape:  (444, 60)
2
X_shuffle shape:  (444, 60)
X_train shape:  (355, 60)
y_train shape:  (355,)
X_test shape:  (89, 60)
y_test shape:  (89,)


2. Defining an RNN. Unlike the previous FNN models, we want to predict the prices in the future
N days based on the prices in the past M days (including today). Usually, M is much larger than N. Here,
you may simply set N to be 1 and choose an integer that is at least 50 for M. You are required to define a
neural network which has at least one recurrent layer for this purpose.

In [19]:
# TODO implement the baseline model

class _RNN_(nn.Module):
    def __init__(self, input_size = 1, hidden_size = 64, output_size = 1):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first = True)
        self.fully_connected_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        output, hidden_state = self.rnn(x)
        output = output[:, -1, :]
        output = self.fully_connected_layer(output)
        return output

def batch_data(X_train : np.ndarray, y_train : np.ndarray, n_batches = 10):
    X_train, y_train = shuffle(X_train, y_train, random_state=rstate)
    batch_size = int(X_train.shape[0] / n_batches)
    # this split code was taken from google
    X_batches = [X_train[i: i + batch_size] for i in range(0, X_train.shape[0], batch_size)]
    y_batches = [y_train[i: i + batch_size] for i in range(0, y_train.shape[0], batch_size)]

    return X_batches, y_batches

# NOTE the data is scaled, so we need to perform the inverse transform to map it back. 
def train_model(model, X_train, y_train, X_test, y_test, epochs=8):
    mean_squared_error = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

    losses = []

    training_start_time = time.time()

    X_test_t = torch.tensor(X_test, dtype = torch.float32).unsqueeze(-1)
    y_test_t = torch.tensor(y_test, dtype = torch.float32).unsqueeze(-1)

    for epoch in range(epochs):
        epoch_loss = 0

        # batch data for each epoch
        X_batches, y_batches = batch_data(X_train, y_train, n_batches = 10)

        # extra inner loop to cover each minibatch
        # float64 is apparently not okay?
        # model expects a 3D tensor, (n_batches, n_samples, n_features), must unsqueeze.

        # training mode
        model.train()

        batch_iters = 0
        for X_batch, y_batch in zip(X_batches, y_batches):
            X_batch_t = torch.tensor(X_batch, dtype = torch.float32).unsqueeze(-1)
            y_batch_t = torch.tensor(y_batch, dtype = torch.float32).unsqueeze(-1)

            # clear gradients
            optimizer.zero_grad()
            predictions = model(X_batch_t)
            loss = mean_squared_error(predictions, y_batch_t)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            batch_iters += 1

        # must average the loss for batches.
        print("train loss on epoch ", epoch, ": ", epoch_loss / np.size(X_batch_t, 0))
        losses.append(epoch_loss / np.size(X_batch_t, 0))

        with torch.no_grad():
            test_predictions = model(X_test_t)
            test_loss = mean_squared_error(test_predictions, y_test_t)
            print("test loss on epoch ",  epoch,  ": ",  test_loss) 

    training_end_time = time.time()

    training_time = training_end_time - training_start_time

    model.eval()
    

    # y_mean = y_train.mean()
    # print("y_train_mean: ", y_train.mean())
    # print("mse mean: ", ((y_test - y_mean)**2).mean())

    # y_last = X_test[:, -1]
    # print("y_last: ", y_last)
    # print("mse_last: ", ((y_last - y_test)**2).mean())
    
    return {
        "test_loss": test_loss.item(),
        "train_losses": losses,
        "training_time": training_time
    }

In [76]:
# create a new RNN for each 

print("GPU? ", torch.cuda.is_available())
print(torch.get_num_threads())

spy_RNN = _RNN_(1, 16, 1)
ko_RNN = _RNN_(1, 16, 1)
wmt_RNN = _RNN_(1, 16, 1)
ed_RNN = _RNN_(1, 16, 1)

nets = [spy_RNN, ko_RNN, wmt_RNN, ed_RNN]

# train_model(spy_RNN, prepped_stocks[0]["X_train"], prepped_stocks[0]["y_train"], prepped_stocks[0]["X_test"], prepped_stocks[0]["y_test"])

# train_model(ko_RNN, prepped_stocks[1]["X_train"], prepped_stocks[1]["y_train"], prepped_stocks[1]["X_test"], prepped_stocks[1]["y_test"])

# train_model(wmt_RNN, prepped_stocks[2]["X_train"], prepped_stocks[2]["y_train"], prepped_stocks[2]["X_test"], prepped_stocks[2]["y_test"])

train_model(ed_RNN, prepped_stocks[3]["X_train"], prepped_stocks[3]["y_train"], prepped_stocks[3]["X_test"], prepped_stocks[3]["y_test"])



GPU?  False
8
train loss on epoch  0 :  1.2970790982246398
test loss on epoch  0 :  tensor(0.4972)
train loss on epoch  1 :  0.8987619757652283
test loss on epoch  1 :  tensor(0.3362)
train loss on epoch  2 :  0.587870517373085
test loss on epoch  2 :  tensor(0.2010)
train loss on epoch  3 :  0.324103482067585
test loss on epoch  3 :  tensor(0.0811)
train loss on epoch  4 :  0.10178548227995635
test loss on epoch  4 :  tensor(0.0167)
train loss on epoch  5 :  0.05032810661941767
test loss on epoch  5 :  tensor(0.0204)
train loss on epoch  6 :  0.0495153620839119
test loss on epoch  6 :  tensor(0.0145)
train loss on epoch  7 :  0.04470679871737957
test loss on epoch  7 :  tensor(0.0146)


{'test_loss': 0.014619751833379269,
 'train_losses': [1.2970790982246398,
  0.8987619757652283,
  0.587870517373085,
  0.324103482067585,
  0.10178548227995635,
  0.05032810661941767,
  0.0495153620839119,
  0.04470679871737957],
 'training_time': 0.32189488410949707}